In [2]:
!pip install stable_baselines3 gymnasium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.0/184.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 958.1/958.1 kB 24.5 MB/s eta 0:00:00


In [3]:
import numpy as np
from stable_baselines3 import TD3
import gymnasium as gym
from stable_baselines3.common.noise import NormalActionNoise


In [5]:
env = gym.make("Pendulum-v1", render_mode="rgb_array")
print("Environment Details:")
print(f"Action Space: {env.action_space}")
print(f"Observation Space: {env.observation_space}")
print(f"Action Space Low: {env.action_space.low}")
print(f"Action Space High: {env.action_space.high}")
print(f"Observation Space Low: {env.observation_space.low}")
print(f"Observation Space High: {env.observation_space.high}")

print("\nEnvironment Metadata:")
print(env.metadata)

obs, info = env.reset()
print("\nInitial Observation:", obs)

Environment Details:
Action Space: Box(-2.0, 2.0, (1,), float32)
Observation Space: Box([-1. -1. -8.], [1. 1. 8.], (3,), float32)
Action Space Low: [-2.]
Action Space High: [2.]
Observation Space Low: [-1. -1. -8.]
Observation Space High: [1. 1. 8.]

Environment Metadata:
{'render_modes': ['human', 'rgb_array'], 'render_fps': 30}

Initial Observation: [-0.19342569 -0.9811149  -0.30205432]


In [9]:
n_actions = env.action_space.shape[-1]
action_noise = NormalActionNoise(
              mean=np.zeros(n_actions),
              sigma=0.1 * np.ones(n_actions),
              )
model = TD3(
    env = env,
    policy= "MlpPolicy",
    action_noise = action_noise,
    buffer_size = 200000,
    learning_starts = 10000,
    batch_size = 256,
    tau = 0.005,
    gamma = 0.98,
    train_freq = 2,
    gradient_steps = -1,
    optimize_memory_usage = False,
    target_policy_noise = 0.2,
    target_noise_clip = 0.5,
    policy_delay = 2,
    learning_rate = 0.001,
    seed = 42,
    policy_kwargs = dict(net_arch=[400, 300]),
    )

In [10]:
print("Starting training...")
model.learn(total_timesteps=500000, progress_bar=True)
print("Training complete.")


  19% ━━━━━━━━━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96,515/500,000  [ 1:06:26 < 5:39:23 , 20 it/s ]

KeyboardInterrupt: 

In [ ]:
save_path = "."
model.save(save_path)
print(f"Model saved at {save_path}")

In [ ]:
loaded_model = TD3.load(save_path, env=env)
print(f"Model loaded from {save_path}")

In [ ]:
total_rewards = []

episode_num = 10
max_stagnant_steps = 5
episode_max_steps = 200

for episode in range(episode_num):
        obs, _ = env.reset()
        episode_reward = 0

        previous_reward = None
        stagnant_steps = 0

        for i in range(episode_max_steps):
          env.render()

          action, _ = model.predict(obs, deterministic=True)
          obs, reward, done, _, _ = env.step(action)
          episode_reward += reward

          if reward == previous_reward:
              stagnant_steps += 1
          else:
              stagnant_steps = 0

          previous_reward = reward

          if stagnant_steps >= max_stagnant_steps:
                print(f"Breaking out due to stagnant reward after {stagnant_steps} steps")
                break

          if done:
            break
        total_rewards.append(episode_reward)
        print(f"Episode {episode + 1:<3} Reward: {episode_reward:>5.2f}")

mean_reward = np.mean(total_rewards)
std_reward = np.std(total_rewards)
print(f"Evaluation complete. Mean Reward: {mean_reward:.2f}, Std Reward: {std_reward:.2f}")